In [ ]:
!pip install datasets -q
print("Done!")

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Downloading...")
ds = load_dataset("artem9k/ai-text-detection-pile", split="train")
df = ds.to_pandas()

print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
print(df['source'].value_counts())

In [ ]:
human_df = df[df['source'] == 'human'][['text']].copy()
human_df['label'] = 0

ai_df = df[df['source'] == 'ai'][['text']].copy()
ai_df['label'] = 1

print(f"Available human : {len(human_df)}")
print(f"Available AI    : {len(ai_df)}")

n = min(25000, len(human_df), len(ai_df))
balanced = pd.concat([
    human_df.sample(n, random_state=42),
    ai_df.sample(n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

balanced = balanced[['text', 'label']]
print(f"\nTotal : {len(balanced)}")

In [ ]:
print("=== Balance Check ===")
print(balanced['label'].value_counts())
print(f"\nHuman : {(balanced['label']==0).sum()}")
print(f"AI    : {(balanced['label']==1).sum()}")
print(f"Ratio : {(balanced['label']==0).sum() / (balanced['label']==1).sum():.2f}")

In [ ]:
# Save for use in your model notebooks
balanced.to_csv('ai_pile_balanced.csv', index=False)
print("Saved! Ready for training.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('ai_pile_balanced.csv', '/content/drive/MyDrive/ai_pile_balanced.csv')
print("Saved to Drive!")

In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy -q
print("Done!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ai_pile_balanced.csv')

print(df.shape)
print(df.head())
print(df['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

df = df[['text', 'label']].dropna()
df['text'] = df['text'].astype(str)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['label']
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df['text'])
val_encodings = tokenize(val_df['text'])
test_encodings = tokenize(test_df['text'])

print("Tokenization completed!")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TextDataset(train_encodings, train_df['label'])
val_dataset = TextDataset(val_encodings, val_df['label'])
test_dataset = TextDataset(test_encodings, test_df['label'])

print("Dataset created!")

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("BERT model loaded!")

In [ ]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("GPU not enabled")

In [ ]:
from transformers import Trainer, TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    logging_first_step=True,
    disable_tqdm=False,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["Human", "AI"]))

print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))

In [ ]:
model_path = "/content/drive/MyDrive/bert_ai_detector"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("BERT model saved!")

In [ ]:
print("trainer" in globals())
print("model" in globals())
print("test_dataset" in globals())

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_text(text):
    if len(text.split()) < 10:
        print("⚠️ Text too short. Please enter longer text.")
        return

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    label = "👤 Human Written" if pred == 0 else "🤖 AI Generated"

    print("Prediction:", label)
    print(f"Confidence: {confidence * 100:.2f}%")

while True:
    text = input("\nEnter text or type quit: ")

    if text.lower() == "quit":
        print("Exiting...")
        break

    predict_text(text)